# Tasks 6 & 7: Feature Selection and Predictive Model

This notebook defines the features for predicting total piece travel time, trains an XGBoost Regressor, and evaluates its performance.

### The prediction problem

**Goal**: Predict the total travel time to the quench bath (`lifetime_bath_s`) using only data available **early in the process** — before the piece has finished the line.

**Why this is useful**: If we can predict the total time after only the 2nd strike, we can raise real-time alerts for pieces that are likely to be slow, allowing operators to investigate the cause while the piece is still on the line.

### Feature selection rationale

**Constraint**: the model must predict using only information available **after the 2nd strike** — approximately 18 seconds into the ~58-second journey. Any feature that requires waiting for later stages cannot be used, because by the time those values exist, the prediction is no longer useful.

#### Selected features

| Feature | Type | Available at | Why include it |
|---|---|---|---|
| `die_matrix` | Categorical (int) | Before processing | Each die has different tooling geometry and expected times — matrix 4974 has a median bath time of ~56s while 5091 has ~59s |
| `lifetime_2nd_strike_s` | Continuous (seconds) | After 2nd strike (~18s) | The earliest cumulative time — if the piece is already slow here, it will likely carry that delay through to the bath |
| `oee_cycle_time_s` | Continuous (seconds) | Rolling metric | Production rate context — slower OEE may correlate with systematic delays (robot trajectory adjustments, hydraulic pressure drops) |

#### Excluded features

| Feature | Why excluded |
|---|---|
| `lifetime_3rd_strike_s` | Available too late (~25s) — the piece is already halfway through the main press |
| `lifetime_4th_strike_s` | Available too late (~38s) — and has ~16% missing data from a sensor offline period |
| `lifetime_auxiliary_press_s` | Available too late (~55s) — only ~2s before the bath, prediction would be useless |
| `lifetime_general_s` | Equivalent to `lifetime_bath_s` — redundant with the target |
| `partial_*` columns | Derived from cumulative times that include late-stage data |
| `piece_id` | Not predictive — just an identifier |

In [1]:
import json
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

GOLD_FILE  = Path('../data/gold/pieces.parquet')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

FEATURES = ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
TARGET   = 'lifetime_bath_s'
OEE_MEDIAN = 13.8   # default for missing OEE

pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup complete.')


Setup complete.


## 1. Load gold dataset

In [2]:
df = pd.read_parquet(GOLD_FILE)
print(f'Gold dataset: {len(df):,} rows')
print(f'Target range: {df[TARGET].min():.1f}s – {df[TARGET].max():.1f}s')
print(f'OEE null: {df["oee_cycle_time_s"].isna().sum():,} ({100*df["oee_cycle_time_s"].isna().mean():.1f}%)')
display(df[FEATURES + [TARGET]].describe().round(2))


Gold dataset: 167,679 rows
Target range: 43.9s – 74.7s
OEE null: 42,185 (25.2%)


,die_matrix,lifetime_2nd_strike_s,oee_cycle_time_s,lifetime_bath_s
count,167679.0000,167679.0000,125494.0000,167679.0000
mean,5074.8200,18.6400,13.8800,58.6500
std,34.5800,2.3200,0.6000,3.2800
min,4974.0000,9.4000,11.0000,43.9000
25%,5090.0000,16.9000,13.4700,56.3000
50%,5090.0000,18.0000,13.8100,58.3000
75%,5091.0000,19.8000,14.2100,60.2000
max,5091.0000,31.0000,16.0000,74.7000


## 2. Prepare features and target

- **Features (X)**: `die_matrix`, `lifetime_2nd_strike_s`, `oee_cycle_time_s`
- **Target (y)**: `lifetime_bath_s`

Drop rows where any feature or the target is NULL (missing drill data does not affect us here since we don't use drill as a feature).

In [3]:
# Fill missing OEE with median before dropping — keeps more rows
df['oee_cycle_time_s'] = df['oee_cycle_time_s'].fillna(OEE_MEDIAN)

model_df = df[FEATURES + [TARGET]].dropna().copy()
X = model_df[FEATURES]
y = model_df[TARGET]

print(f'Rows available for modelling: {len(model_df):,}')
print(f'Features: {FEATURES}')
print(f'Target: {TARGET}')
display(X.head(3))


Rows available for modelling: 167,679
Features: ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
Target: lifetime_bath_s


,die_matrix,lifetime_2nd_strike_s,oee_cycle_time_s
0,5052,17.9000,13.8000
1,5052,17.9000,13.8000
2,5052,18.2000,13.8000


## 3. Feature correlation with target

How strongly does each feature correlate with the total bath time? High correlation suggests predictive value.

In [4]:
corr = model_df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
print('Pearson correlation with lifetime_bath_s:')
print(corr.round(4).to_string())
print('\nInterpretation:')
print('  lifetime_2nd_strike_s: strong positive — early delays carry through to bath')
print('  die_matrix:            weak linear — non-linear relationship handled by XGBoost')
print('  oee_cycle_time_s:      weak-moderate — production rate context')


Pearson correlation with lifetime_bath_s:
die_matrix              0.2266
lifetime_2nd_strike_s   0.7659
oee_cycle_time_s        0.3180

Interpretation:
  lifetime_2nd_strike_s: strong positive — early delays carry through to bath
  die_matrix:            weak linear — non-linear relationship handled by XGBoost
  oee_cycle_time_s:      weak-moderate — production rate context


## 4. Train/test split

Split 80/20 with a fixed random seed for reproducibility. Stratify by die_matrix to ensure each matrix is represented proportionally in both sets.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=model_df['die_matrix']
)
print(f'Train: {len(X_train):,} pieces')
print(f'Test:  {len(X_test):,} pieces')
print('\nMatrix distribution in train set:')
print((X_train['die_matrix'].value_counts(normalize=True)*100).round(1))


Train: 134,143 pieces
Test:  33,536 pieces

Matrix distribution in train set:
die_matrix
5090   48.7000
5091   29.6000
5052   12.5000
4974    9.3000
Name: proportion, dtype: float64


## 5. Train XGBoost Regressor

XGBoost is chosen because:
- Handles mixed feature types (categorical die_matrix + continuous times)
- Robust to the remaining noise in the data
- Fast training on ~100k rows
- Produces feature importance rankings

In [6]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('Model trained.')


Model trained.


## 6. Evaluate on test set

Key metrics:
- **RMSE**: root mean squared error (same unit as target — seconds)
- **MAE**: mean absolute error (average prediction error in seconds)
- **R²**: coefficient of determination (1.0 = perfect, 0.0 = no better than mean)

In [7]:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('Test set metrics:')
print(f'  RMSE: {rmse:.3f}s')
print(f'  MAE:  {mae:.3f}s')
print(f'  R²:   {r2:.4f}')
print(f'\nContext: typical bath time ~58s → MAE represents ±{100*mae/y_test.median():.1f}% error')
print('Goal is to detect multi-second delays — sub-1s precision is sufficient.')


Test set metrics:
  RMSE: 1.872s
  MAE:  0.941s
  R²:   0.6706

Context: typical bath time ~58s → MAE represents ±1.6% error
Goal is to detect multi-second delays — sub-1s precision is sufficient.


## 7. Performance per die matrix

Check if the model performs equally well across all matrices, or if some are harder to predict.

In [8]:
test_df = X_test.copy()
test_df['actual']    = y_test.values
test_df['predicted'] = y_pred

matrix_metrics = []
for matrix, grp in test_df.groupby('die_matrix'):
    matrix_metrics.append({
        'die_matrix': int(matrix),
        'n':    len(grp),
        'RMSE': round(root_mean_squared_error(grp['actual'], grp['predicted']), 3),
        'MAE':  round(mean_absolute_error(grp['actual'], grp['predicted']), 3),
        'R2':   round(r2_score(grp['actual'], grp['predicted']), 4),
    })

metrics_df = pd.DataFrame(matrix_metrics).set_index('die_matrix')
display(metrics_df)
print('\nModel should perform consistently across all matrices.')


,n,RMSE,MAE,R2
die_matrix,,,,
4974,3106,0.9030,0.4670,0.7392
5052,4177,1.5170,0.7620,0.6864
5090,16336,2.0770,1.1260,0.6249
5091,9917,1.8740,0.8590,0.6742



Model should perform consistently across all matrices.


## 8. Feature importance

Which features contribute most to the prediction? This validates the feature selection rationale.

In [9]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Feature importance (gain-based):')
print(importance.round(4).to_string())

chart = alt.Chart(
    importance.reset_index().rename(columns={'index':'feature', 0:'importance'})
).mark_bar().encode(
    x=alt.X('importance:Q', title='Importance (gain)'),
    y=alt.Y('feature:N', sort='-x', title='Feature'),
    tooltip=['feature:N', 'importance:Q']
).properties(title='XGBoost feature importance', width=400, height=150)
chart


Feature importance (gain-based):
lifetime_2nd_strike_s   0.7366
oee_cycle_time_s        0.1888
die_matrix              0.0746


alt.Chart(...)

## 9. Save model and metadata

Save the trained model and its metadata for use by the inference service (Task 8).

In [10]:
model_path    = MODELS_DIR / 'xgboost_bath_predictor.json'
metadata_path = MODELS_DIR / 'model_metadata.json'

model.save_model(str(model_path))

metadata = {
    'features':       FEATURES,
    'target':         TARGET,
    'oee_median':     OEE_MEDIAN,
    'valid_matrices': sorted(int(m) for m in model_df['die_matrix'].unique()),
    'training_rows':  int(len(X_train)),
    'test_rows':      int(len(X_test)),
    'hyperparameters': {
        'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1,
        'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42
    },
    'metrics': {
        'rmse': round(float(rmse), 4),
        'mae':  round(float(mae), 4),
        'r2':   round(float(r2), 4),
    },
    'per_matrix_metrics': {str(r['die_matrix']): {
        'rmse': r['RMSE'], 'mae': r['MAE'], 'r2': r['R2']
    } for r in matrix_metrics},
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Model saved:    {model_path}')
print(f'Metadata saved: {metadata_path}')
print(json.dumps(metadata, indent=2))


Model saved:    ../models/xgboost_bath_predictor.json
Metadata saved: ../models/model_metadata.json
{
  "features": [
    "die_matrix",
    "lifetime_2nd_strike_s",
    "oee_cycle_time_s"
  ],
  "target": "lifetime_bath_s",
  "oee_median": 13.8,
  "valid_matrices": [
    4974,
    5052,
    5090,
    5091
  ],
  "training_rows": 134143,
  "test_rows": 33536,
  "hyperparameters": {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
  },
  "metrics": {
    "rmse": 1.8716,
    "mae": 0.9407,
    "r2": 0.6706
  },
  "per_matrix_metrics": {
    "4974": {
      "rmse": 0.903,
      "mae": 0.467,
      "r2": 0.7392
    },
    "5052": {
      "rmse": 1.517,
      "mae": 0.762,
      "r2": 0.6864
    },
    "5090": {
      "rmse": 2.077,
      "mae": 1.126,
      "r2": 0.6249
    },
    "5091": {
      "rmse": 1.874,
      "mae": 0.859,
      "r2": 0.6742
    }
  }
}


## 10. Summary

In [11]:
print('=' * 60)
print('PREDICTIVE MODEL SUMMARY')
print('=' * 60)
print(f'  Target:   lifetime_bath_s (~58s typical)')
print(f'  Features: {FEATURES}')
print(f'  Constraint: only data available after 2nd strike (~18s)')
print()
print(f'  RMSE: {rmse:.3f}s  |  MAE: {mae:.3f}s  |  R²: {r2:.4f}')
print(f'  MAE = {100*mae/y_test.median():.1f}% of typical bath time')
print()
print('  Feature importance order:')
for feat, imp in importance.items():
    print(f'    {feat}: {imp:.4f}')
print()
print(f'  Model artifact: models/xgboost_bath_predictor.json')
print(f'  Metadata:       models/model_metadata.json')
print('=' * 60)


PREDICTIVE MODEL SUMMARY
  Target:   lifetime_bath_s (~58s typical)
  Features: ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
  Constraint: only data available after 2nd strike (~18s)

  RMSE: 1.872s  |  MAE: 0.941s  |  R²: 0.6706
  MAE = 1.6% of typical bath time

  Feature importance order:
    lifetime_2nd_strike_s: 0.7366
    oee_cycle_time_s: 0.1888
    die_matrix: 0.0746

  Model artifact: models/xgboost_bath_predictor.json
  Metadata:       models/model_metadata.json
